In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Force interactive window
%matplotlib qt 

# 1. Create Dummy 3D Data (Time, Y, X)
n_frames, size = 200, 128
time = np.linspace(0, 1, n_frames)
data_3d = np.random.normal(0, 0.2, (n_frames, size, size))
data_3d[:, 40:60, 40:60] += np.sin(2 * np.pi * 15 * time)[:, None, None]

mean_img = np.mean(data_3d, axis=0)

# 2. Setup Visualization with radius as a mutable list for global access
radius = [10]  # Using a list to allow modification inside functions
last_mouse_pos = [0, 0] # Store last known position to refresh on scroll

fig, (ax_img, ax_freq) = plt.subplots(1, 2, figsize=(12, 5))
ax_img.imshow(mean_img, cmap='viridis', origin='lower')
circle_patch = Circle((0, 0), radius[0], color='white', fill=False, lw=2)
ax_img.add_patch(circle_patch)

freq_line, = ax_freq.plot([], [], color='firebrick')
ax_freq.set_title("Temporal Frequency (FFT of Circle Sum)")
ax_freq.set_xlabel("Frequency (Hz)")

def update_analysis(cx, cy):
    """Core logic to update ROI and FFT based on position and current radius."""
    cx, cy = int(cx), int(cy)
    circle_patch.set_center((cx, cy))
    circle_patch.set_radius(radius[0])
    
    y_idx, x_idx = np.ogrid[:size, :size]
    mask = (x_idx - cx)**2 + (y_idx - cy)**2 <= radius[0]**2
    
    if np.any(mask):
        time_series = data_3d[:, mask].mean(axis=1)
        yf = np.fft.rfft(time_series - np.mean(time_series))
        xf = np.fft.rfftfreq(len(time_series), d=(time[1]-time[0]))
        
        freq_line.set_data(xf, np.abs(yf))
        ax_freq.set_xlim(0, xf.max())
        ax_freq.set_ylim(0, np.max(np.abs(yf)) * 1.1 + 0.1)
    fig.canvas.draw_idle()

def on_move(event):
    if event.inaxes != ax_img: return
    last_mouse_pos[0], last_mouse_pos[1] = event.xdata, event.ydata
    update_analysis(event.xdata, event.ydata)

def on_scroll(event):
    """Changes the circle radius when the mouse wheel is scrolled."""
    if event.inaxes != ax_img: return
    
    # Adjust radius: event.step is +1 for up, -1 for down
    radius[0] = max(2, radius[0] + int(event.step * 2)) 
    
    # Update analysis immediately at current mouse position
    update_analysis(last_mouse_pos[0], last_mouse_pos[1])

# Connect both motion and scroll events
fig.canvas.mpl_connect('motion_notify_event', on_move)
fig.canvas.mpl_connect('scroll_event', on_scroll)

plt.tight_layout()
plt.show()
